In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.utils.class_weight import compute_class_weight

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, classification_report
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

In [ ]:
def build_area_hour_category_feature_file(
    file_path,
    output_path=None,
    time_freq="h",
    use_full_day_grid=True,
    top_n_categories=None
):
    """
    Build a processed area-hour feature table for crime category prediction.

    Output unit:
        one row per (Community Area, hour_slot)

    Target:
        - crime_category = most frequent Primary Type in that area-hour
        - NO_CRIME if no incident occurred in that area-hour
    """

    # 1. Load only required columns
    df = pd.read_csv(file_path, usecols=["Date", "Community Area", "Primary Type"])

    # 2. Parse and clean
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df["Community Area"] = pd.to_numeric(df["Community Area"], errors="coerce")
    df["Primary Type"] = df["Primary Type"].astype(str).str.strip()

    df = df.dropna(subset=["Date", "Community Area", "Primary Type"]).copy()
    df["Community Area"] = df["Community Area"].astype(int)

    # Optional: keep only top N categories, merge others into OTHER
    if top_n_categories is not None:
        top_types = df["Primary Type"].value_counts().nlargest(top_n_categories).index
        df["Primary Type"] = df["Primary Type"].where(df["Primary Type"].isin(top_types), "OTHER")

    # 3. Create hourly time slot
    df["hour_slot"] = df["Date"].dt.floor(time_freq)

    # 4. Aggregate crime count per area-hour
    crime_counts = (
        df.groupby(["Community Area", "hour_slot"])
          .size()
          .reset_index(name="crime_count")
    )
    crime_counts["crime_occurrence"] = (crime_counts["crime_count"] > 0).astype(int)

    # 5. Find dominant crime type in each area-hour
    type_counts = (
        df.groupby(["Community Area", "hour_slot", "Primary Type"])
          .size()
          .reset_index(name="type_count")
    )

    dominant_type = (
        type_counts.sort_values(
            ["Community Area", "hour_slot", "type_count", "Primary Type"],
            ascending=[True, True, False, True]
        )
        .drop_duplicates(subset=["Community Area", "hour_slot"], keep="first")
        [["Community Area", "hour_slot", "Primary Type"]]
        .rename(columns={"Primary Type": "crime_category"})
    )

    # 6. Create full area-hour grid
    all_areas = sorted(df["Community Area"].unique())

    if use_full_day_grid:
        start_day = df["Date"].dt.floor("D").min()
        end_day = df["Date"].dt.floor("D").max()

        all_hours = pd.date_range(
            start=start_day,
            end=end_day + pd.Timedelta(hours=23),
            freq=time_freq
        )
    else:
        all_hours = pd.date_range(
            start=df["hour_slot"].min(),
            end=df["hour_slot"].max(),
            freq=time_freq
        )

    full_grid = pd.MultiIndex.from_product(
        [all_areas, all_hours],
        names=["Community Area", "hour_slot"]
    ).to_frame(index=False)

    # 7. Merge counts and dominant category into full grid
    data = full_grid.merge(
        crime_counts,
        on=["Community Area", "hour_slot"],
        how="left"
    )

    data = data.merge(
        dominant_type,
        on=["Community Area", "hour_slot"],
        how="left"
    )

    # Fill zero-crime rows
    data["crime_count"] = data["crime_count"].fillna(0)
    data["crime_occurrence"] = data["crime_occurrence"].fillna(0).astype(int)
    data["crime_category"] = data["crime_category"].fillna("NO_CRIME")

    # 8. Time features
    data["hour"] = data["hour_slot"].dt.hour
    data["day_of_week"] = data["hour_slot"].dt.dayofweek
    data["month"] = data["hour_slot"].dt.month
    data["day"] = data["hour_slot"].dt.day
    data["is_weekend"] = (data["day_of_week"] >= 5).astype(int)

    # 9. Cyclical hour encoding
    data["hour_sin"] = np.sin(2 * np.pi * data["hour"] / 24)
    data["hour_cos"] = np.cos(2 * np.pi * data["hour"] / 24)

    # 10. Sort before lag/rolling features
    data = data.sort_values(["Community Area", "hour_slot"]).reset_index(drop=True)

    # 11. Lag features from past crime_count
    grouped = data.groupby("Community Area")["crime_count"]

    data["lag_1h"] = grouped.shift(1)
    data["lag_2h"] = grouped.shift(2)
    data["lag_3h"] = grouped.shift(3)
    data["lag_24h"] = grouped.shift(24)

    # 12. Rolling features using past values only
    shifted = grouped.shift(1)

    data["rolling_3h_mean"] = shifted.rolling(3).mean().reset_index(level=0, drop=True)
    data["rolling_6h_mean"] = shifted.rolling(6).mean().reset_index(level=0, drop=True)
    data["rolling_12h_mean"] = shifted.rolling(12).mean().reset_index(level=0, drop=True)
    data["rolling_24h_mean"] = shifted.rolling(24).mean().reset_index(level=0, drop=True)

    # 13. Fill missing lag/rolling values
    lag_roll_cols = [
        "lag_1h", "lag_2h", "lag_3h", "lag_24h",
        "rolling_3h_mean", "rolling_6h_mean",
        "rolling_12h_mean", "rolling_24h_mean"
    ]
    data[lag_roll_cols] = data[lag_roll_cols].fillna(0)

    # 14. Type cleanup
    int_cols = [
        "Community Area", "crime_occurrence",
        "hour", "day_of_week", "month", "day", "is_weekend"
    ]
    for col in int_cols:
        data[col] = data[col].astype(int)

    # 15. Save if requested
    if output_path is not None:
        data.to_csv(output_path, index=False)

    return data

In [ ]:
from google.colab import files

# Upload file
uploaded = files.upload()

# Get uploaded file name
file_path = list(uploaded.keys())[0]

In [ ]:
processed_category_df = build_area_hour_category_feature_file(
    file_path=file_path)

print(processed_category_df.shape)
print(processed_category_df.columns)
print(processed_category_df["crime_category"].value_counts())
processed_category_df.head()

In [ ]:
processed_category_df = processed_category_df.sort_values(by="hour_slot").reset_index(drop=True)

In [ ]:
processed_category_df.head()

In [ ]:
processed_category_df = processed_category_df[
    processed_category_df["crime_category"] != "NO_CRIME"
].copy()

print(processed_category_df["crime_category"].value_counts())

In [ ]:
def map_crime_group(x):

    if x in [
        "THEFT", "BURGLARY", "MOTOR VEHICLE THEFT",
        "CRIMINAL DAMAGE", "DECEPTIVE PRACTICE"
    ]:
        return "PROPERTY_CRIME"

    elif x in [
        "BATTERY", "ASSAULT", "ROBBERY",
        "HOMICIDE", "CRIMINAL SEXUAL ASSAULT"
    ]:
        return "VIOLENT_CRIME"

    elif x in [
        "NARCOTICS", "WEAPONS VIOLATION"
    ]:
        return "DRUGS_WEAPONS"

    else:
        return "OTHER_CRIME"

In [ ]:
processed_category_df["crime_group"] = processed_category_df["crime_category"].apply(map_crime_group)

print(processed_category_df["crime_group"].value_counts())

In [ ]:
# target column
target_col = "crime_group"



# feature columns
feature_cols = [
    "Community Area",
    "day_of_week",
    "month",
    "day",
    "is_weekend",
    "hour_sin",
    "hour_cos",
    "lag_1h",
    "lag_2h",
    "lag_3h",
    "lag_24h",
    "rolling_3h_mean",
    "rolling_6h_mean",
    "rolling_12h_mean",
    "rolling_24h_mean",
    "crime_occurrence"
]

In [ ]:
model_df = processed_category_df[feature_cols + [target_col]].copy()

# separate features and target
X = model_df[feature_cols]
y = model_df[target_col]

# train-test split
# shuffle=False keeps chronological order
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=False
)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# check shapes
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

In [ ]:
y.value_counts()

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

rf_model = RandomForestClassifier(
    n_estimators=50,        # reduced trees → less RAM
    max_depth=12,           # control tree size
    class_weight="balanced",
    n_jobs=-1,              # use all CPU cores
    random_state=42
)

rf_model.fit(X_train_scaled, y_train)

y_pred_rf = rf_model.predict(X_test_scaled)

print("Random Forest Results")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

In [ ]:
from imblearn.over_sampling import RandomOverSampler
from collections import Counter

# Check original class counts
class_counts = Counter(y_train)
print("Before:", class_counts)

# Set a maximum count for each minority class
# This avoids RAM crash
max_count = 5000   # you can change to 3000, 5000, or 10000

sampling_strategy = {
    cls: min(max_count, max(class_counts.values()))
    for cls, count in class_counts.items()
    if count < max_count
}

print("Sampling strategy:", sampling_strategy)

ros = RandomOverSampler(
    sampling_strategy=sampling_strategy,
    random_state=42
)

X_train_ros, y_train_ros = ros.fit_resample(
    X_train_scaled,
    y_train
)

print("After:", Counter(y_train_ros))
print("Balanced shape:", X_train_ros.shape)

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, accuracy_score

nn_model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    max_iter=200,
    random_state=42
)

nn_model.fit(X_train_ros, y_train_ros)

y_pred_nn = nn_model.predict(X_test_scaled)

print("Neural Network Results")
print("Accuracy:", accuracy_score(y_test, y_pred_nn))
print(classification_report(y_test, y_pred_nn))

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import pandas as pd

comparison = pd.DataFrame({
    "Model": ["Random Forest", "Neural Network"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_nn)
    ],
    "Macro Precision": [
        precision_score(y_test, y_pred_rf, average="macro", zero_division=0),
        precision_score(y_test, y_pred_nn, average="macro", zero_division=0)
    ],
    "Macro Recall": [
        recall_score(y_test, y_pred_rf, average="macro", zero_division=0),
        recall_score(y_test, y_pred_nn, average="macro", zero_division=0)
    ],
    "Macro F1": [
        f1_score(y_test, y_pred_rf, average="macro", zero_division=0),
        f1_score(y_test, y_pred_nn, average="macro", zero_division=0)
    ],
    "Weighted F1": [
        f1_score(y_test, y_pred_rf, average="weighted", zero_division=0),
        f1_score(y_test, y_pred_nn, average="weighted", zero_division=0)
    ]
})

comparison

In [ ]:
print(comparison)

In [ ]:
best_model = comparison.sort_values("Macro F1", ascending=False).iloc[0]

print("Best model is:", best_model["Model"])
print("Macro F1:", best_model["Macro F1"])

In [ ]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, accuracy_score, f1_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
lstm_df = processed_category_df.copy()

lstm_df = lstm_df.sort_values(
    ["Community Area", "hour_slot"]
).reset_index(drop=True)

lstm_df = lstm_df.head(50000)

In [ ]:
le = LabelEncoder()

lstm_df["target_encoded"] = le.fit_transform(lstm_df[target_col])

num_classes = len(le.classes_)

print("Number of classes:", num_classes)
print("Classes:", le.classes_)

In [ ]:
time_steps = 3

X_seq = []
y_seq = []

for area in lstm_df["Community Area"].unique():
    area_data = lstm_df[lstm_df["Community Area"] == area]

    X_vals = area_data[feature_cols].values
    y_vals = area_data["target_encoded"].values

    for i in range(time_steps, len(area_data)):
        X_seq.append(X_vals[i-time_steps:i])
        y_seq.append(y_vals[i])

X_seq = np.array(X_seq, dtype="float32")
y_seq = np.array(y_seq, dtype="int32")

X_seq = np.nan_to_num(X_seq)

print("X_seq shape:", X_seq.shape)
print("y_seq shape:", y_seq.shape)

In [ ]:
split = int(len(X_seq) * 0.8)

X_train_lstm = X_seq[:split]
X_test_lstm = X_seq[split:]

y_train_lstm = y_seq[:split]
y_test_lstm = y_seq[split:]

print("X_train_lstm:", X_train_lstm.shape)
print("X_test_lstm:", X_test_lstm.shape)

In [ ]:
# -----------------------------
# Scale LSTM data
# -----------------------------

scaler_lstm = StandardScaler()

num_features = X_train_lstm.shape[2]

X_train_2d = X_train_lstm.reshape(-1, num_features)
X_test_2d = X_test_lstm.reshape(-1, num_features)

X_train_scaled_2d = scaler_lstm.fit_transform(X_train_2d)
X_test_scaled_2d = scaler_lstm.transform(X_test_2d)

X_train_lstm_scaled = X_train_scaled_2d.reshape(X_train_lstm.shape).astype("float32")
X_test_lstm_scaled = X_test_scaled_2d.reshape(X_test_lstm.shape).astype("float32")

print("Scaled train shape:", X_train_lstm_scaled.shape)

In [ ]:
# -----------------------------
# Class weights for imbalance
# -----------------------------

classes = np.unique(y_train_lstm)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_lstm
)

class_weights = dict(zip(classes, weights))

print("Class weights:", class_weights)

In [ ]:
# -----------------------------
# Build improved LSTM model
# -----------------------------

lstm_model = Sequential([
    LSTM(32, input_shape=(time_steps, num_features)),
    Dropout(0.2),

    Dense(32, activation="relu"),
    Dropout(0.3),

    Dense(num_classes, activation="softmax")
])

lstm_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

lstm_model.summary()

In [ ]:
# -----------------------------
# Train LSTM with Early Stopping
# -----------------------------

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history = lstm_model.fit(
    X_train_lstm_scaled,
    y_train_lstm,
    epochs=30,
    batch_size=32,
    validation_split=0.2,
    class_weight=class_weights,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# -----------------------------
# Predict with LSTM
# -----------------------------

y_pred_probs_lstm = lstm_model.predict(X_test_lstm_scaled)

y_pred_lstm = np.argmax(y_pred_probs_lstm, axis=1)

print("LSTM prediction created")
print(y_pred_lstm[:10])

In [ ]:
# -----------------------------
# Evaluate LSTM
# -----------------------------

lstm_accuracy = accuracy_score(y_test_lstm, y_pred_lstm)
lstm_macro_f1 = f1_score(
    y_test_lstm,
    y_pred_lstm,
    average="macro",
    zero_division=0
)

print("LSTM Accuracy:", lstm_accuracy)
print("LSTM Macro F1:", lstm_macro_f1)

print(classification_report(
    y_test_lstm,
    y_pred_lstm,
    zero_division=0
))

In [ ]:
from sklearn.metrics import precision_score, recall_score

comparison = pd.DataFrame({
    "Model": [
        "Random Forest",
        "Neural Network (MLP)",
        "LSTM"
    ],
    "Accuracy": [
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_nn),
        accuracy_score(y_test_lstm, y_pred_lstm)
    ],
    "Macro Precision": [
        precision_score(y_test, y_pred_rf, average="macro", zero_division=0),
        precision_score(y_test, y_pred_nn, average="macro", zero_division=0),
        precision_score(y_test_lstm, y_pred_lstm, average="macro", zero_division=0)
    ],
    "Macro Recall": [
        recall_score(y_test, y_pred_rf, average="macro", zero_division=0),
        recall_score(y_test, y_pred_nn, average="macro", zero_division=0),
        recall_score(y_test_lstm, y_pred_lstm, average="macro", zero_division=0)
    ],
    "Macro F1": [
        f1_score(y_test, y_pred_rf, average="macro", zero_division=0),
        f1_score(y_test, y_pred_nn, average="macro", zero_division=0),
        f1_score(y_test_lstm, y_pred_lstm, average="macro", zero_division=0)
    ]
})

comparison